In [ ]:
import pandas as pd
import nbformat

In [ ]:
import pandas as pd
import requests

# -----------------------------
# API URLs
# -----------------------------
urls = {
    "Obesity_adults": "https://ghoapi.azureedge.net/api/NCD_BMI_30C",
    "Obesity_children": "https://ghoapi.azureedge.net/api/NCD_BMI_PLUS2C",
    "Malnutrition_adults": "https://ghoapi.azureedge.net/api/NCD_BMI_18C",
    "Malnutrition_children": "https://ghoapi.azureedge.net/api/NCD_BMI_MINUS2C"
}

# -----------------------------
# Function to fetch API safely
# -----------------------------
def fetch_api_data(url):
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()  # Raises error for 4xx/5xx

        if response.text.strip():   # Ensure response not empty
            return response.json()
        else:
            print(f"Empty response from: {url}")
            return {}

    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return {}

# -----------------------------
# Fetch all API data
# -----------------------------
api_data = {}

for key, url in urls.items():
    print(f"Fetching: {key}")
    api_data[key] = fetch_api_data(url)

# -----------------------------
# Convert to DataFrames
# -----------------------------
dataframes = {}

for key, data in api_data.items():

    # Safely extract 'value'
    df = pd.DataFrame(data.get('value', [])) # "Give me the value of key 'value',If it does NOT exist, return an empty list [] instead"

    # Add age group column
    if "adults" in key:
        df["age_group"] = "Adults"
    else:
        df["age_group"] = "Children"

    # Add category column (Obesity / Malnutrition)
    if "Obesity" in key:
        df["category"] = "Obesity"
    else:
        df["category"] = "Malnutrition"

    dataframes[key] = df

# -----------------------------
# Access individual DataFrames
# -----------------------------
Obesity_adults_df = dataframes["Obesity_adults"]
Obesity_children_df = dataframes["Obesity_children"]
Malnutrition_adults_df = dataframes["Malnutrition_adults"]
Malnutrition_children_df = dataframes["Malnutrition_children"]

print("Data loaded successfully ✅")


In [ ]:
Obesity_adults_df

In [ ]:
Obesity_children_df

In [ ]:
Malnutrition_adults_df

In [ ]:
Malnutrition_children_df

Combining  the two dataset into one

In [ ]:
Obesity_df = pd.merge(Obesity_adults_df,Obesity_children_df,how='outer')

In [ ]:
Obesity_df

In [ ]:
Malnutrition_df = pd.merge(Malnutrition_adults_df,Malnutrition_children_df,how='outer')

In [ ]:
Malnutrition_df

Filter each dataset to include only records from the years 2012 to 2022

In [ ]:
Obesity_df1 = Obesity_df[(Obesity_df['TimeDimensionValue'] >= '2012') & (Obesity_df['TimeDimensionValue'] <= '2022')]

In [ ]:
Obesity_df1.info()

In [ ]:
Malnutrition_df1 = Malnutrition_df[(Malnutrition_df['TimeDimensionValue'] >= '2012') & (Malnutrition_df['TimeDimensionValue'] <= '2022')]

In [ ]:
Malnutrition_df1.info()

Step 2: 🧹 Data Cleaning & Feature Engineering

Columns to Retain:
Only keep the following columns from each dataset:
* ParentLocation
* Dim1
* TimeDim
* Low
* High
* NumericValue
* SpatialDim
* age_group



In [ ]:
Total_columns = list(Obesity_df1.columns)
len(Total_columns)

In [ ]:
requried = ['ParentLocation','Dim1','TimeDim', 'Low', 'High', 'NumericValue','SpatialDim', 'age_group']
not_requried = []
requried_1 = []
for i in Total_columns:
  if i not in requried:
    not_requried.append(i)
  else:
    requried_1.append(i)
print(not_requried)
print(requried_1)

In [ ]:
Obesity_df1.drop(columns=not_requried,axis=1,inplace= True)

In [ ]:
Obesity_df1

In [ ]:
Malnutrition_df1.drop(columns=not_requried,axis=1,inplace=True)

In [ ]:
Malnutrition_df1

 Rename Columns for Consistency:


In [ ]:
Obesity_df1.rename(columns={'SpatialDim':'Country','ParentLocation':'Region','TimeDim':'Year','Dim1':'Gender','NumericValue':'Mean_Estimate','Low':'LowerBound','High':'UpperBound','age_group':'Age_Group'},inplace=True)

In [ ]:
Malnutrition_df1.rename(columns={'SpatialDim':'Country','ParentLocation':'Region','TimeDim':'Year','Dim1':'Gender','NumericValue':'Mean_Estimate','Low':'LowerBound','High':'UpperBound','age_group':'Age_Group'},inplace=True)

In the Gender column, standardize values to 'Male', 'Female', and 'Both'

In [ ]:
Obesity_df1['Gender']=Obesity_df1['Gender'].replace({'SEX_BTSX':'Both','SEX_MLE':"Male","SEX_FMLE":"Female"})

In [ ]:
Malnutrition_df1['Gender']=Malnutrition_df1['Gender'].replace({'SEX_BTSX':'Both','SEX_MLE':"Male","SEX_FMLE":"Female"})

Convert Country Codes to Full Names using pycountry:

In [ ]:
! pip install pycountry

In [ ]:
import pycountry
import numpy as np

In [ ]:
aland = pycountry.countries.get(alpha_3='SYC')
n = aland.name
print(n)

In [ ]:
# v = Obesity_df1['Country'].unique()
# z=[]
# for i in v:
#   country_code = str(i)
#   country_obj = pycountry.countries.get(alpha_3=country_code)
#   if country_obj:
#     n = country_obj.name
#     z.append(n)
#   else:
#     # If pycountry can't find a match (e.g., for regions or invalid codes),
#     # append the original code.
#     z.append(country_code)
# country1 = np.array(z)

In [ ]:
# unique_codes = Obesity_df1['Country'].unique()
# special_cases = {
#                     'GLOBAL': 'Global',
#                     'WB_LMI': 'Low & Middle Income',
#                     'WB_HI': 'High Income',
#                     'WB_LI': 'Low Income',
#                     'EMR': 'Eastern Mediterranean Region',
#                     'EUR': 'Europe',
#                     'AFR': 'Africa',
#                     'SEAR': 'South-East Asia Region',
#                     'WPR': 'Western Pacific Region',
#                     'AMR': 'Americas Region',
#                     'WB_UMI': 'Upper Middle Income'}
# z=[]
# for i in unique_codes:
#   country_code = str(i)
#   country_name = pycountry.countries.get(alpha_3=country_code)
#   if country_name:
#     n = country_name.name
#     z.append(n)
#   else:
#     if country_code in special_cases:
#         z.append(special_cases[country_code])
#     else:
#         z.append(country_code) # Keep original if no mapping found
# country = np.array(z)

In [ ]:
# unique_country_codes_for_mapping = Obesity_df1['Country'].unique()
# mapping_dict = dict(zip(unique_country_codes_for_mapping, country))
# Obesity_df1['Country'] = Obesity_df1['Country'].replace(mapping_dict)

In [ ]:
# unique_codes = Obesity_df1['Country'].unique()
# special_cases = {
#                     'GLOBAL': 'Global',
#                     'WB_LMI': 'Low & Middle Income',
#                     'WB_HI': 'High Income',
#                     'WB_LI': 'Low Income',
#                     'EMR': 'Eastern Mediterranean Region',
#                     'EUR': 'Europe',
#                     'AFR': 'Africa',
#                     'SEAR': 'South-East Asia Region',
#                     'WPR': 'Western Pacific Region',
#                     'AMR': 'Americas Region',
#                     'WB_UMI': 'Upper Middle Income'}
# z=[]
# for i in unique_codes:
#   country_code = str(i)
#   country_name = pycountry.countries.get(alpha_3=country_code)
#   if country_name:
#     n = country_name.name
#     z.append(n)
#   else:
#     if country_code in special_cases:
#         z.append(special_cases[country_code])
#     else:
#         z.append(country_code) # Keep original if no mapping found
# country = np.array(z)

In [ ]:
def extract_country_name(unique_codes,special_cases):
  z=[]
  for i in unique_codes:
    country_code = str(i)
    country_name = pycountry.countries.get(alpha_3=country_code)
    if country_name:
      n = country_name.name
      z.append(n)
    else:
      if country_code in special_cases:
        z.append(special_cases[country_code])
      else:
        z.append(country_code) # Keep original if no mapping found
  country = np.array(z)
  return country

code1 = Obesity_df1['Country'].unique()
code2 = Malnutrition_df1['Country'].unique()

special_code =  {
                    'GLOBAL': 'Global',
                    'WB_LMI': 'Low & Middle Income',
                    'WB_HI': 'High Income',
                    'WB_LI': 'Low Income',
                    'EMR': 'Eastern Mediterranean Region',
                    'EUR': 'Europe',
                    'AFR': 'Africa',
                    'SEAR': 'South-East Asia Region',
                    'WPR': 'Western Pacific Region',
                    'AMR': 'Americas Region',
                    'WB_UMI': 'Upper Middle Income'}
codes = ['code1','code2']
for i in codes:
  if i == 'code1':
    Obesity_country = extract_country_name(code1, special_code)
  else:
    Malnutrition_country = extract_country_name(code2, special_code)

In [ ]:
unique_country_codes_for_mapping = Obesity_df1['Country'].unique()
mapping_dict = dict(zip(unique_country_codes_for_mapping, Obesity_country))
Obesity_df1['Country'] = Obesity_df1['Country'].replace(mapping_dict)

In [ ]:
unique_codes = Malnutrition_df1['Country'].unique()
Malnutrition_mapping_dict = dict(zip(unique_codes,Malnutrition_country))
Malnutrition_df1['Country']=Malnutrition_df1['Country'].replace(Malnutrition_mapping_dict)

In [ ]:
Obesity_df1.info()

Outlier Dectition

In [ ]:
Obesity_df1['Mean_Estimate'].describe()

In [ ]:
Q1 = np.percentile(Obesity_df1['Mean_Estimate'],25)
Q2 = np.percentile(Obesity_df1['Mean_Estimate'],50)
Q3 = np.percentile(Obesity_df1['Mean_Estimate'],75)

In [ ]:
IQR = Q3 - Q1
lower_bound = Q1 - (1.5*IQR)
print(round(lower_bound,2))
upper_bound = Q3 + (1.5*IQR)
print(round(upper_bound,2))


In [ ]:
Obesity_df1 = Obesity_df1[(Obesity_df1['Mean_Estimate'] >= lower_bound)& (Obesity_df1['Mean_Estimate'] <= upper_bound) ]

In [ ]:
import plotly.express as px
px.box(data_frame=Obesity_df1['Mean_Estimate'])

In [ ]:
Malnutrition_df1['Mean_Estimate'].describe()

In [ ]:
Q1 = np.percentile(Malnutrition_df1['Mean_Estimate'],25)
Q2 = np.percentile(Malnutrition_df1['Mean_Estimate'],50)
Q3 = np.percentile(Malnutrition_df1['Mean_Estimate'],75)

In [ ]:
IQR = Q3 - Q1
upper_bound = Q3 + (1.5*IQR)
print(round(upper_bound,2))
lower_bound = Q1 - (1.5*IQR)
print(round(lower_bound,2))

In [ ]:
Malnutrition_df1 = Malnutrition_df1[(Malnutrition_df1['Mean_Estimate'] > lower_bound) & (Malnutrition_df1['Mean_Estimate'] < upper_bound) ]

In [ ]:
import plotly.express as px
px.box(data_frame=Malnutrition_df1['Mean_Estimate'])


In [ ]:
Obesity_df1['LowerBound'].describe()

In [ ]:
Q1 = np.percentile(Obesity_df1['LowerBound'],25)
Q2 = np.percentile(Obesity_df1['LowerBound'],50)
Q3 = np.percentile(Obesity_df1['LowerBound'],75)

IQR = Q3-Q1
lower_bound = Q1 - (1.5*IQR)
print(lower_bound)
upper_bound = Q3 + (1.5*IQR)
print(upper_bound)

In [ ]:
Obesity_df1 = Obesity_df1[(Obesity_df1['LowerBound'] >= lower_bound)& (Obesity_df1['LowerBound'] <= upper_bound)]

In [ ]:
px.box(data_frame=Obesity_df1['LowerBound'])

In [ ]:
Obesity_df1['UpperBound'].describe()

In [ ]:
Q1 = np.percentile(Obesity_df1['UpperBound'],25)
Q2 = np.percentile(Obesity_df1['UpperBound'],50)
Q3 = np.percentile(Obesity_df1['UpperBound'],75)

IQR = Q3-Q1
lower_bound = Q1 - (1.5*IQR)
print(lower_bound)
upper_bound = Q3 + (1.5*IQR)
print(upper_bound)

In [ ]:
Obesity_df1 = Obesity_df1[(Obesity_df1['UpperBound'] >= lower_bound)& (Obesity_df1['UpperBound'] <= upper_bound)]

In [ ]:
px.box(data_frame=Obesity_df1['UpperBound'])

In [ ]:
Malnutrition_df1['LowerBound'].describe()

In [ ]:
Q1 = np.percentile(Malnutrition_df1['LowerBound'],25)
Q2 = np.percentile(Malnutrition_df1['LowerBound'],50)
Q3 = np.percentile(Malnutrition_df1['LowerBound'],75)

IQR = Q3-Q1
lower_bound = Q1 - (1.5*IQR)
print(lower_bound)
upper_bound = Q3 + (1.5*IQR)
print(upper_bound)

In [ ]:
Malnutrition_df1 = Malnutrition_df1[(Malnutrition_df1['LowerBound'] >= lower_bound)& (Malnutrition_df1['LowerBound'] <= upper_bound)]

In [ ]:
px.box(data_frame=Malnutrition_df1['LowerBound'])

In [ ]:
Malnutrition_df1['UpperBound'].describe()

In [ ]:
Q1 = np.percentile(Malnutrition_df1['UpperBound'],25)
Q2 = np.percentile(Malnutrition_df1['UpperBound'],50)
Q3 = np.percentile(Malnutrition_df1['UpperBound'],75)

IQR = Q3-Q1
lower_bound = Q1 - (1.5*IQR)
print(lower_bound)
upper_bound = Q3 + (1.5*IQR)
print(upper_bound)

In [ ]:
Malnutrition_df1 = Malnutrition_df1[(Malnutrition_df1['UpperBound'] >= lower_bound)& (Malnutrition_df1['UpperBound'] <= upper_bound)]

In [ ]:
px.box(data_frame=Malnutrition_df1['UpperBound'])

In [ ]:
Obesity_df1.reset_index(inplace=True)
Obesity_df1.drop(columns='index',inplace=True)

In [ ]:
Obesity_df1.info()

In [ ]:
Malnutrition_df1.reset_index(inplace=True)
Malnutrition_df1.drop(columns='index',inplace=True)

In [ ]:
Malnutrition_df1.info()

New Columns to Create:

In [ ]:
Obesity_df1['CI_Width']  = Obesity_df1['UpperBound'] - Obesity_df1['LowerBound']

In [ ]:
Malnutrition_df1['CI_Width'] = Malnutrition_df1['UpperBound'] -  Malnutrition_df1['LowerBound']

In [ ]:
Obesity_df1copy = Obesity_df1.copy()

In [ ]:
import numpy as np

condition_1 = [
    Obesity_df1['Mean_Estimate'] <= 25,
    (Obesity_df1['Mean_Estimate'] > 25) & (Obesity_df1copy['Mean_Estimate'] < 30),
    Obesity_df1['Mean_Estimate'] >= 30
]

choice_1 = ["Low", "Moderate", "High"]

# Provide an explicit default value that is a string, matching the choices dtype
Obesity_df1['Obesity_level'] = np.select(condition_1, choice_1, default='Unclassified')


In [ ]:
Obesity_df1['Obesity_level'].value_counts()

In [ ]:
Malnutrition_df1copy = Malnutrition_df1.copy()

In [ ]:
condtion_2 = [
    Malnutrition_df1['Mean_Estimate'] <= 10,
    (Malnutrition_df1['Mean_Estimate'] >10) & (Malnutrition_df1['Mean_Estimate']<20),
    Malnutrition_df1['Mean_Estimate']>=20]

choice_2 = ["Low", "Moderate", "High"]

Malnutrition_df1['Malnutrition_level'] = np.select(condtion_2,choice_2,default='Unclassified')

In [ ]:
Malnutrition_df1['Malnutrition_level'].value_counts()

In [ ]:
Obesity_df1.duplicated().sum()

In [ ]:
Malnutrition_df1.duplicated().sum()

In [ ]:
#Obesity_df1.to_csv(r'/content/drive/MyDrive/Data set/Project/Obesity_df1.csv')

In [ ]:
#Malnutrition_df1.to_csv(r'/content/drive/MyDrive/Data set/Project/Malnutrition_df1.csv')

In [ ]:
Obesity_df1.isnull().sum()

In [ ]:
Malnutrition_df1.isnull().sum()

In [ ]:
Obesity_df1.loc[Obesity_df1['Region'].isnull(), ['Country', 'Region','Age_Group','Obesity_level']]

In [ ]:
Obesity_df1['CI_Width'].describe()

In [ ]:
Malnutrition_df1['CI_Width'].describe()

In [ ]:
Obesity_df1.info()

In [ ]:
Malnutrition_df1.info()

In [ ]:
df_grouped = (
    Obesity_df1
    .groupby(['Year', 'Region'])['Mean_Estimate']
    .mean()
    .reset_index()
)

px.line(df_grouped, x='Year', y='Mean_Estimate', color='Region')


In [ ]:
import nbformat
df_grouped = (
    Obesity_df1
    .groupby(['Year', 'Age_Group'])['Mean_Estimate']
    .mean()
    .reset_index()
)

px.line(df_grouped, x='Year', y='Mean_Estimate', color='Age_Group')


In [ ]:
Obesity_df1['Region'].fillna("Europe",inplace=True)

In [ ]:
Malnutrition_df1['Region'].fillna("Europe",inplace=True)